In [2]:
!pip install pandas


[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
from google.colab import files
import numpy as np

ModuleNotFoundError: No module named 'google'

# Step:1 Load and explore the data
    in this step we will:
    - Load the dataset using pandas
    - Check for missing values, datatypes and basic statistics
    - Analyze class distribution (Fraud vs Non-fraud)

In [ ]:
# load and display first few rows of each dataset
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DATA_DIRECTORY = Path("/content/drive/MyDrive/Fraud_Detection/Data")
transactions = pd.read_csv(DATA_DIRECTORY / "credit_card_transactions-ibm_v2.csv")
cards = pd.read_csv(DATA_DIRECTORY / "sd254_cards.csv")
users = pd.read_csv(DATA_DIRECTORY / "sd254_users.csv")

print("Transactions:\n", transactions.head(), "\n")
print("cards:\n", cards.head(), "\n")
print("users:\n", users.head(), "\n")

In [ ]:
# rename the Person column to user in the "users" dataframe
users = users.rename(columns={"Person": "User"})
# creat a new user column with index value starting from 0
users["User"] = range(len(users))
print(users)

In [ ]:
# rename the CARD INDEX column to Card in the "cards" dataset
cards = cards.rename(columns={"CARD INDEX": "Card"})
print(cards)

In [ ]:
print(transactions)

In [ ]:
# merge users and cards with outer join
users_cards = pd.merge(users, cards, how="outer", on="User")

In [ ]:
transactions.shape
# sample the large dataset
# transactions_sample= transactions.sample(n=1000000, random_state=42)

In [ ]:
transactions

In [ ]:
# Stratified sampling by 'Year' and 'User'
# Get unique users from all data
unique_users = transactions["User"].unique()

# Sample 30% of users randomly
np.random.seed(42)
sampled_users = np.random.choice(
    unique_users, size=int(len(unique_users) * 0.3), replace=False
)

# Keep all transactions of sampled users
stratified_sample = transactions[transactions["User"].isin(sampled_users)]

In [ ]:
print("Total Users:", stratified_sample["User"].nunique())
print("Total Transactions:", len(stratified_sample))
print(stratified_sample["Year"].value_counts())

In [ ]:
# merge users and cards with left join
merged_df = stratified_sample.merge(users_cards, on=["User", "Card"], how="left")

In [ ]:
merged_df.shape

In [ ]:
print(merged_df.head())

In [ ]:
print(merged_df.columns)

In [ ]:
print(merged_df.isnull().sum())

In [ ]:
print(merged_df.info())

# Step 2: Data Cleaning


In [ ]:
# filling missing values
merged_df["Errors?"].fillna("No Error", inplace=True)
merged_df.loc[merged_df["Merchant City"].str.lower() == "online", "Merchant State"] = (
    "online"
)
merged_df.loc[merged_df["Merchant City"].str.lower() == "online", "Zip"] = (
    "ONLINE"  # for online purchases
)
merged_df["Zip"] = merged_df["Zip"].apply(
    lambda x: "Unknown"
    if (pd.isna(x) or x == "")
    else str(int(x))
    if isinstance(x, float)
    else x
)  # converting data type to string
print(merged_df["Merchant State"].isnull().sum())
print(merged_df["Zip"].dtype)

### dealing with datatype


In [ ]:
merged_df["Zipcode"] = merged_df["Zipcode"].astype(str)

In [ ]:
# remove the dollar sign and convert to float
merged_df["Amount"] = merged_df["Amount"].replace("[\$,]", "", regex=True).astype(float)
merged_df["Per Capita Income - Zipcode"] = (
    merged_df["Per Capita Income - Zipcode"]
    .replace("[\$,]", "", regex=True)
    .astype(float)
)
merged_df["Yearly Income - Person"] = (
    merged_df["Yearly Income - Person"].replace("[\$,]", "", regex=True).astype(float)
)
merged_df["Total Debt"] = (
    merged_df["Total Debt"].replace("[\$,]", "", regex=True).astype(float)
)
merged_df["Credit Limit"] = (
    merged_df["Credit Limit"].replace("[\$,]", "", regex=True).astype(float)
)

In [ ]:
print(
    f"Amount data type is:{merged_df['Amount'].dtype}\n"
    f"Per Capita Income - Zipcode data type is:{merged_df['Per Capita Income - Zipcode'].dtype}\n"
    f"Yearly Income - Person data type is:{merged_df['Yearly Income - Person'].dtype}\n"
    f"Total Debt data type is:{merged_df['Total Debt'].dtype}\n"
    f"Credit Limit data type is:{merged_df['Credit Limit'].dtype}\n"
)

In [ ]:
# Convert categorical columns to category type
categorical_cols = [
    "Card Type",
    "Gender",
    "Card Brand",
    "Use Chip",
    "Is Fraud?",
    "Has Chip",
    "Card on Dark Web",
]
for col in categorical_cols:
    merged_df[col] = merged_df[col].astype("category")

In [ ]:
merged_df.info()

In [ ]:
merged_df.isnull().sum()

In [ ]:
# Since the dataset is randomly sampled, the timestamps don’t cover a continuous time span. Extracting the hour allows us to analyze fraud patterns based on time (e.g., what hours have the most fraudulent transactions).
merged_df["Transaction Hour"] = pd.to_datetime(
    merged_df["Time"], format="%H:%M"
).dt.hour

In [ ]:
print(merged_df["Transaction Hour"])

In [ ]:
# Convert account open date and expires to proper format
merged_df["Acct Open Date"] = pd.to_datetime(
    merged_df["Acct Open Date"], format="%m/%Y"
)
merged_df["Expires"] = pd.to_datetime(merged_df["Expires"], format="%m/%Y")

In [ ]:
merged_df["Zip"]

In [38]:
# Save cleaned data
merged_df.to_csv("cleaned_data_second_version_.csv", index=False)
print("Cleaned data saved to 'cleaned_data_second_version.csv'")

Cleaned data saved to 'cleaned_data_second_version.csv'


In [39]:
from google.colab import files

files.download("cleaned_data_second_version_.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
merged_df.info()

NameError: name 'merged_df' is not defined